# Reliable RAG：让检索“更可靠”（相关性过滤 + 幻觉检查 + 证据高亮）

<img src="../images/reliable_rag.svg" width="320" />

这一节的目标不是发明新的检索算法，而是把 RAG 变得更“可靠”：

- **检索相关性过滤**：把明显不相关的检索结果丢掉
- **幻觉/不 grounded 检查**：判断生成答案是否被证据支持
- **证据片段高亮**：从被用到的文档中抽出“逐字匹配”的支持片段（方便你做可解释/审计/评估）

## 0) 导入依赖并加载环境变量

你需要确保已安装 `langchain` / `langchain-openai` / `langchain-community` 等。

In [ ]:
import os
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

/tmp/ipykernel_2211521/181449768.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [ ]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

## 1) 构建一个最小向量库（Web → split → FAISS）

原仓库用 DeepLearning.AI 的几篇文章作为语料，我们沿用相同的学习节奏：

- 抓取网页
- 切 chunk
- 建向量库


In [ ]:
urls = [
    "https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-2-reflection/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-4-planning/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=0,
)
doc_splits = text_splitter.split_documents(docs_list)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(doc_splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

len(doc_splits)

13

## 2) 一个问题：先检索


In [ ]:
question = "What are the different kinds of agentic design patterns?"
docs = retriever.invoke(question)
len(docs)

4

## 3) 检查检索结果是否相关（relevance grading）

这一步的目标很朴素：**过滤掉明显跑偏的检索结果**。


In [ ]:
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: Literal["yes", "no"] = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
doc_grader_llm = llm.with_structured_output(
    GradeDocuments,
    method="json_schema",
    strict=True,
)

system = (
    "You are a grader assessing relevance of a retrieved document to a user question. "
    "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. "
    "It does not need to be stringent. The goal is to filter out erroneous retrievals. "
    "Return 'yes' or 'no'."
)

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}"),
    ]
)

retrieval_grader = grade_prompt | doc_grader_llm

In [ ]:
docs_to_use = []
for doc in docs:
    res = retrieval_grader.invoke({"question": question, "document": doc.page_content})
    if res.binary_score == "yes":
        docs_to_use.append(doc)

len(docs_to_use)

4

## 4) 用过滤后的 docs 生成答案（RAG）


In [ ]:
def format_docs(docs):
    return "\n".join(
        f"<doc{i+1}>\nTitle:{doc.metadata.get('title','')}\nSource:{doc.metadata.get('source','')}\nContent:{doc.page_content}\n</doc{i+1}>\n"
        for i, doc in enumerate(docs)
    )


system = (
    "You are an assistant for question-answering tasks. "
    "Answer using the retrieved documents. Use 3-5 sentences maximum and keep it concise."
)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents:\n\n<docs>{documents}</docs>\n\nUser question: <question>{question}</question>"),
    ]
)

generation = (rag_prompt | llm).invoke(
    {"documents": format_docs(docs_to_use), "question": question}
)
generation.content

'The different kinds of agentic design patterns include Tool Use, Reflection, Planning, and Multi-Agent Collaboration. Tool Use and Reflection are more mature and reliable patterns, while Planning and Multi-Agent Collaboration are less predictable but offer exciting capabilities. Each pattern serves to enhance the performance and functionality of AI agents in various tasks.'

## 5) 检查答案是否 grounded（hallucination grading）

这一步只做一个判断：生成答案是否被证据支持。


In [ ]:
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: Literal["yes", "no"] = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


hallucination_grader_llm = llm.with_structured_output(
    GradeHallucinations,
    method="json_schema",
    strict=True,
)

system = (
    "You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. "
    "Return 'yes' if the answer is grounded in the facts; otherwise return 'no'."
)

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts:\n\n<facts>{documents}</facts>\n\nLLM generation: <generation>{generation}</generation>"),
    ]
)

hallucination_grader = hallucination_prompt | hallucination_grader_llm

hallucination_check = hallucination_grader.invoke(
    {"documents": format_docs(docs_to_use), "generation": generation.content}
)
hallucination_check

GradeHallucinations(binary_score='yes')

## 6) 高亮“答案用到的证据片段”（verbatim segments）

目标：从 docs 里抽出**逐字匹配**的片段，能直接支撑答案。


In [ ]:
class HighlightSpan(BaseModel):
    doc_id: str = Field(description="Document id like D1, D2")
    title: str = Field(description="Document title")
    source: str = Field(description="Document source")
    segments: list[str] = Field(description="Verbatim snippets from the document")


class HighlightDocuments(BaseModel):
    highlights: list[HighlightSpan] = Field(description="Evidence highlights")


highlight_llm = llm.with_structured_output(
    HighlightDocuments,
    method="json_schema",
    strict=True,
)

docs_block = "\n\n".join(
    [
        f"<doc id='D{i+1}'>\nTitle:{d.metadata.get('title','')}\nSource:{d.metadata.get('source','')}\nContent:{d.page_content}\n</doc>"
        for i, d in enumerate(docs_to_use)
    ]
)

system = (
    "You are a document evidence extractor. "
    "Given a question, an answer, and a set of documents, extract verbatim snippets from the documents that directly support the answer. "
    "Each snippet must be an exact substring of the document content. "
    "Only include documents that were actually used."
)

highlight_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Question:\n{question}\n\nAnswer:\n{answer}\n\nDocuments:\n{documents}"),
    ]
)

lookup_response = (highlight_prompt | highlight_llm).invoke(
    {
        "question": question,
        "answer": generation.content,
        "documents": docs_block,
    }
)

lookup_response

HighlightDocuments(highlights=[HighlightSpan(doc_id='D2', title='Agentic Design Patterns Part 3: Tool Use', source='https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io', segments=['Tool Use and Reflection, which I described in last week’s letter, are design patterns that I can get to work fairly reliably on my applications — both are capabilities well worth learning about.', 'In future letters, I’ll describe the Planning and Multi-agent collaboration design patterns. They allow AI agents to do much more but are less mature, less predictable — albeit very exciting — technologies.']), HighlightSpan(doc_id='D4', title='Agentic Design Patterns Part 5, Multi-Agent Collaboration', source='https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io', segments=['multi-agent collaboration is the last of the four key AI agentic design patterns that I’ve described in rece